In [8]:
%pip install playwright
!python -m playwright install chromium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


|                                                                                |   0% of 172.8 MiB
|■■■■■■■■                                                                        |  10% of 172.8 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

(node:25872) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
(node:17680) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
(node:8748) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
(node:10228) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to

In [12]:
from playwright.async_api import async_playwright
import csv
import asyncio
import sys

url = "https://vn.investing.com/currencies/usd-vnd-historical-data"

toggle = ".flex.flex-1.items-center.gap-3\\.5.rounded.border.border-solid.border-\\[\\#CFD4DA\\].bg-white.px-3\\.5.py-2.shadow-select"
date_inputs = ".NativeDateInputV2_root__uAIu0 input[type='date']"
apply_btn = ".flex.cursor-pointer.items-center.gap-3.rounded.bg-v2-blue.py-2\\.5.pl-4.pr-6.shadow-button"

table_rows = ".freeze-column-w-1 tbody tr"

async def scrape_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        await page.goto(url, wait_until="domcontentloaded")

        # open date picker
        await page.locator(toggle).click()
        await page.wait_for_selector(date_inputs)

        inputs = page.locator(date_inputs)

        # start date
        await inputs.nth(0).fill("2022-01-01")

        # end date (today on page)
        await inputs.nth(1).fill("2026-03-06")

        # apply
        await page.locator(apply_btn).click()

        # wait for data to refresh
        await page.wait_for_timeout(20000)

        rows = page.locator(table_rows)
        row_count = await rows.count()

        data = []
        for i in range(row_count):
            cols = rows.nth(i).locator("td")
            col_count = await cols.count()
            row = [await cols.nth(j).inner_text() for j in range(col_count)]
            row.pop(-2)  # remove empty KL column (second last)
            data.append(row)

        await browser.close()
        return data

def run_playwright_in_new_loop():
    # On Windows/Jupyter, use Proactor loop in a worker thread so subprocess works.
    if sys.platform.startswith("win"):
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()

    try:
        asyncio.set_event_loop(loop)
        return loop.run_until_complete(scrape_data())
    finally:
        loop.close()

# Run Playwright in a background thread with its own event loop.
data = await asyncio.to_thread(run_playwright_in_new_loop)

# save csv
with open("usd_vnd_history.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Date", "Price", "Open", "High", "Low", "Change %"])
    writer.writerows(data)

print(f"Saved {len(data)} rows to usd_vnd_history.csv")

Saved 1092 rows to usd_vnd_history.csv


In [14]:
%pip install pandas
import pandas as pd

# 1. Đọc dữ liệu từ file CSV (Thay 'ten_file_cua_ban.csv' bằng tên file thực tế)
df = pd.read_csv("usd_vnd_history.csv")

# 2. LÀM SẠCH DỮ LIỆU SỐ (Numeric Cleaning)
# Xóa dấu ngoặc kép ", dấu phẩy và ép về kiểu float
cols_to_clean = ['Price', 'Open', 'High', 'Low']
for col in cols_to_clean:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('"', '').str.replace(',', '').astype(float)

# Xóa dấu %, dấu + ở cột Change % và ép về kiểu float
if 'Change %' in df.columns:
    df['Change %'] = df['Change %'].astype(str).str.replace('"', '').str.replace('%', '').str.replace('+', '').astype(float)

# 3. LÀM SẠCH THỜI GIAN & SẮP XẾP LẠI (Time Series Formatting)
# Ép cột Date về chuẩn định dạng thời gian của Pandas
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

# Sắp xếp lại dữ liệu từ CŨ NHẤT đến MỚI NHẤT (Bắt buộc đối với Time Series)
df = df.sort_values('Date').reset_index(drop=True)

# 4. LOẠI BỎ THỨ 7, CHỦ NHẬT (Trường hợp web có lẫn vào)
# Trong Pandas, dayofweek: Thứ 2 = 0, ..., Thứ 6 = 4, Thứ 7 = 5, Chủ nhật = 6
# Lọc chỉ lấy các ngày từ Thứ 2 đến Thứ 6 (< 5)
df = df[df['Date'].dt.dayofweek < 5].reset_index(drop=True)

# ------------------------------------------------------------------
# 5. KỸ THUẬT TẠO ĐẶC TRƯNG (FEATURE ENGINEERING) 
# Định nghĩa: 1 tuần = 5 dòng dữ liệu liên tiếp (5 ngày giao dịch)
# ------------------------------------------------------------------

# Đặc trưng trễ: Giá đóng cửa của 1 ngày giao dịch trước đó (Lag 1)
df['Lag_1'] = df['Price'].shift(1)

# Đặc trưng Đường trung bình động (SMA)
df['SMA_1_Week'] = df['Price'].rolling(window=5).mean()  # Trung bình 1 tuần (5 ngày)
df['SMA_2_Weeks'] = df['Price'].rolling(window=10).mean() # Trung bình 2 tuần (10 ngày)
df['SMA_1_Month'] = df['Price'].rolling(window=20).mean() # Trung bình 1 tháng (~20 ngày giao dịch)

# Độ biến động (Volatility): Độ lệch chuẩn của giá trong 1 tuần qua
df['Volatility_1_Week'] = df['Price'].rolling(window=5).std()

# Xem thử 10 dòng cuối cùng (hiện tại) để kiểm tra kết quả
print(df.tail(10))

  Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl (9.9 MB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


           Date    Price     Open     High      Low  Change %    Lag_1  \
1080 2026-02-23  26120.0  25970.0  26127.0  25948.5      0.58  25970.0   
1081 2026-02-24  26195.0  26129.5  26224.5  26115.0      0.29  26120.0   
1082 2026-02-25  26102.0  26198.5  26202.0  26087.5     -0.36  26195.0   
1083 2026-02-26  26075.0  26100.0  26159.5  26039.0     -0.10  26102.0   
1084 2026-02-27  26045.0  26059.5  26105.0  25998.0     -0.12  26075.0   
1085 2026-03-02  26165.0  26041.5  26232.0  26041.5      0.46  26045.0   
1086 2026-03-03  26200.0  26210.0  26232.0  26143.5      0.13  26165.0   
1087 2026-03-04  26220.0  26244.5  26255.0  26181.0      0.08  26200.0   
1088 2026-03-05  26215.0  26207.0  26256.5  26170.0     -0.02  26220.0   
1089 2026-03-06  26240.0  26228.0  26250.0  26215.0      0.10  26215.0   

      SMA_1_Week  SMA_2_Weeks  SMA_1_Month  Volatility_1_Week  
1080     26000.0      25978.5     25987.15          67.082039  
1081     26045.0      26009.5     25990.20         106.06